In [2]:
!pip install requests librosa audiomentations tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.1.0
    Uninstalling soxr-1.1.0:
      Successfully uninstalled soxr-1.1.0


In [3]:
import os
import requests
import tempfile
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
from collections import Counter
from audiomentations import Compose, AddGaussianNoise, PitchShift, Shift
import csv
import shutil
import pandas as pd
# Essa lista vai guardar os metadados de todas as aves durante o processo
TABELA_METADADOS_GLOBAL = []

## 1. CONFIGURAÇÕES PRINCIPAIS E API

In [4]:
API_KEY = "a266159e5cc437278f820f18bbfa0180e95583e8"

URL_API = "https://xeno-canto.org/api/3/recordings"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [5]:
from google.colab import drive
drive.mount('/content/drive')
PASTA_SAIDA = '/content/drive/MyDrive/Dataset_Aves_Brasil'
ARQUIVO_ESPECIES = 'especies_alvo_lista.csv'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
TAXA_AMOSTRAGEM = 22050
DURACAO_JANELA_SEG = 5
TAMANHO_JANELA_AMOSTRAS = DURACAO_JANELA_SEG * TAXA_AMOSTRAGEM
LIMIAR_SILENCIO_DB = 20
ALVO_POR_ESPECIE = 200 # Total de imagens desejadas por espécie

In [7]:
# Configuração do Data Augmentation para evitar dados desbalanceados
augmenter = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    Shift(p=0.5)
])

## 2. DESCOBERTA DAS ESPÉCIES (API v3)

In [8]:
def obter_top_especies(quantidade=250, minimo_gravacoes=45):
    print(f"Mapeando aves com no mínimo {minimo_gravacoes} gravações...")

    QUERY_BASE = 'cnt:brazil grp:1 lat:"<-20" id?:no'
    parametros = {'query': QUERY_BASE, 'key': API_KEY}
    resposta = requests.get(URL_API, params=parametros, headers=HEADERS)

    if resposta.status_code != 200:
         print(f"Erro na API. Código: {resposta.status_code}")
         return []

    dados = resposta.json()
    try:
        num_paginas = int(dados.get('numPages', 1))
    except:
        num_paginas = 1

    contador_especies = Counter()
    nomes_populares = {} # Dicionário para guardar o nome popular de cada espécie
    especies_ignoradas = ["Sonus naturalis", "Mystery mystery"]

    for pagina in tqdm(range(1, num_paginas + 1), desc="Lendo base de dados"):
        try:
            param_pag = {'query': QUERY_BASE, 'page': pagina, 'key': API_KEY}
            resp_pag = requests.get(URL_API, params=param_pag, headers=HEADERS)

            if resp_pag.status_code == 200:
                gravacoes = resp_pag.json().get('recordings', [])
                for gravacao in gravacoes:
                    # Verifica a licença ANTES de contar a ave!
                    licenca = gravacao.get('lic', '').lower()
                    if 'nd' in licenca:
                        continue # Pula e ignora este áudio se for proibido

                    nome_cientifico = f"{gravacao['gen']} {gravacao['sp']}"

                    # Captura o nome popular (Inglês) da API. Se não vier, marca como Desconhecido
                    nome_popular = gravacao.get('en', 'Desconhecido')

                    if nome_cientifico not in especies_ignoradas and "Mystery" not in nome_cientifico:
                        contador_especies[nome_cientifico] += 1

                        # Salva o nome popular no dicionário associado ao nome científico
                        if nome_cientifico not in nomes_populares:
                            nomes_populares[nome_cientifico] = nome_popular

            import time
            time.sleep(0.5)
        except Exception as e:
            continue

    especies_filtradas = {esp: cont for esp, cont in contador_especies.items() if cont >= minimo_gravacoes}
    contador_filtrado = Counter(especies_filtradas)

    # Agora a lista de top espécies será uma lista de TUPLAS: [(Cientifico, Popular), (Cientifico, Popular)...]
    top_especies = [
        (especie, nomes_populares[especie])
        for especie, count in contador_filtrado.most_common(quantidade)
    ]

    print(f"\nForam encontradas {len(top_especies)} espécies válidas.")
    return top_especies

In [9]:
def carregar_ou_buscar_especies(quantidade=250, minimo_gravacoes=45):
    # Se o arquivo CSV já existe, carrega as duas colunas dele
    if os.path.exists(ARQUIVO_ESPECIES):
        print(f"Arquivo '{ARQUIVO_ESPECIES}' encontrado. Carregando...")
        especies = []
        with open(ARQUIVO_ESPECIES, 'r', encoding='utf-8') as f:
            leitor = csv.reader(f)
            next(leitor) # Pula a primeira linha (cabeçalho)
            for linha in leitor:
                if len(linha) >= 2:
                    especies.append((linha[0], linha[1])) # (Nome Científico, Nome Popular)
        return especies

    # Se não existe, busca na API e cria o arquivo CSV com cabeçalhos
    especies = obter_top_especies(quantidade=quantidade, minimo_gravacoes=minimo_gravacoes)
    with open(ARQUIVO_ESPECIES, 'w', encoding='utf-8', newline='') as f:
        escritor = csv.writer(f)
        escritor.writerow(["Nome Cientifico", "Nome Popular (Ingles)"]) # Cabeçalho do arquivo

        for nome_cientifico, nome_popular in especies:
            escritor.writerow([nome_cientifico, nome_popular])

    return especies

## 3. DOWNLOAD E PROCESSAMENTO DE ÁUDIO

In [10]:
def buscar_gravacoes_cascata(nome_cientifico):
    todas_gravacoes = []
    partes = nome_cientifico.strip().split(" ")

    if len(partes) >= 2:
        genero = partes[0]
        especie = partes[1]
        base_query = f'gen:{genero} sp:{especie} cnt:brazil grp:1 lat:"<-20" id?:no'
    else:
        print(f"   -> Erro: Formato de nome inválido para a API v3: '{nome_cientifico}'")
        return []

    for qualidade in ["A", "B", "C"]:
        query = f'{base_query} q:{qualidade}'
        parametros = {'query': query, 'key': API_KEY}
        resposta = requests.get(URL_API, params=parametros, headers=HEADERS)

        if resposta.status_code == 200:
            gravacoes = resposta.json().get('recordings', [])

            for gravacao in gravacoes:
                licenca = gravacao.get('lic', '').lower()
                if 'nd' in licenca:
                    continue

                # ✅ CORREÇÃO: Adiciona apenas a gravação atual (singular)
                todas_gravacoes.append(gravacao)

            if len(todas_gravacoes) >= 200:
                break
        else:
            print(f"   -> Falha na API (Qualidade {qualidade}): Status {resposta.status_code}")

    return todas_gravacoes[:200]

In [11]:
def extrair_janelas_originais(gravacoes):
    janelas_audio = []

    for gravacao in gravacoes:
        if len(janelas_audio) >= ALVO_POR_ESPECIE:
            break

        url_download = gravacao['file']
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=True) as tmp_mp3:
            try:
                # O download do áudio não precisa da API Key, mas precisa do Header
                resposta = requests.get(url_download, headers=HEADERS)
                tmp_mp3.write(resposta.content)
                tmp_mp3.flush()

                y, sr = librosa.load(tmp_mp3.name, sr=TAXA_AMOSTRAGEM)
                y_trim, _ = librosa.effects.trim(y, top_db=LIMIAR_SILENCIO_DB)

                for inicio in range(0, len(y_trim), TAMANHO_JANELA_AMOSTRAS):
                    fim = inicio + TAMANHO_JANELA_AMOSTRAS
                    pedaco = y_trim[inicio:fim]

                    if len(pedaco) < TAMANHO_JANELA_AMOSTRAS:
                        pedaco = np.pad(pedaco, (0, TAMANHO_JANELA_AMOSTRAS - len(pedaco)))

                    janelas_audio.append(pedaco)
            except Exception as e:
                continue

    return janelas_audio

In [12]:
def balancear_dataset(janelas_audio):
    qtd_atual = len(janelas_audio)
    if qtd_atual == 0: return []

    janelas_finais = list(janelas_audio)
    if qtd_atual >= ALVO_POR_ESPECIE:
        return janelas_finais[:ALVO_POR_ESPECIE]

    faltam = ALVO_POR_ESPECIE - qtd_atual
    print(f"   -> Encontrados {qtd_atual} pedaços. Gerando {faltam} mutantes por IA...")

    for i in range(faltam):
        audio_base = janelas_audio[i % qtd_atual]
        audio_mutante = augmenter(samples=audio_base, sample_rate=TAXA_AMOSTRAGEM)
        janelas_finais.append(audio_mutante)

    return janelas_finais

## 4. GERAÇÃO E SALVAMENTO DE IMAGENS

In [13]:
# Adicionamos o 'id_audio' aqui nos parâmetros da função
def salvar_espectrogramas(janelas_audio, nome_especie, id_audio):
    pasta_especie = os.path.join(PASTA_SAIDA, nome_especie.replace(" ", "_"))
    os.makedirs(pasta_especie, exist_ok=True)

    for idx, audio_array in enumerate(janelas_audio):
        espectrograma = librosa.feature.melspectrogram(y=audio_array, sr=TAXA_AMOSTRAGEM, n_mels=128, fmax=11025)
        espectrograma_db = librosa.power_to_db(espectrograma, ref=np.max)

        plt.figure(figsize=(8, 4))
        librosa.display.specshow(espectrograma_db, sr=TAXA_AMOSTRAGEM, x_axis='time', y_axis='mel', fmax=11025)
        plt.axis('off')

        # Exemplo: XC123456_part00.png
        nome_arquivo = f"XC{id_audio}_part{idx:02d}.png"
        caminho_imagem = os.path.join(pasta_especie, nome_arquivo)

        plt.savefig(caminho_imagem, bbox_inches='tight', pad_inches=0, transparent=True)
        plt.close()

In [1]:
def processar_especie(nome_cientifico):
    pasta_especie = os.path.join(PASTA_SAIDA, nome_cientifico.replace(" ", "_"))

    # VERIFICAÇÃO DE RETOMADA
    if os.path.exists(pasta_especie):
        quantidade_imagens = len([f for f in os.listdir(pasta_especie) if f.endswith('.png')])
        if quantidade_imagens >= ALVO_POR_ESPECIE:
            print(f"\n[PULANDO] {nome_cientifico} - Já possui {quantidade_imagens} imagens prontas.")
            return
        else:
            print(f"\n[REFAZENDO] {nome_cientifico} - Incompleto ({quantidade_imagens} imagens). Recomeçando...")
            shutil.rmtree(pasta_especie)

    print(f"\nIniciando: {nome_cientifico}")
    gravacoes = buscar_gravacoes_cascata(nome_cientifico)

    if not gravacoes:
        print(f"Nenhuma gravação encontrada.")
        return

    # SALVANDO OS METADADOS PARA O KAGGLE
    # ==========================================
    for gravacao in gravacoes:
        TABELA_METADADOS_GLOBAL.append({
            "id_gravacao": f"XC{gravacao.get('id')}",
            "especie_cientifico": nome_cientifico,
            "tipo_canto": gravacao.get('type', 'Desconhecido'),
            "qualidade": gravacao.get('q', 'Desconhecido'),
            "latitude": gravacao.get('lat', ''),
            "longitude": gravacao.get('lng', ''),
            "autor": gravacao.get('rec', 'Desconhecido'),
            "fundo": ", ".join(gravacao.get('also', [])) if isinstance(gravacao.get('also'), list) else gravacao.get('also', '')
        })
    # ==========================================
    # Como a extração original perde o ID individual de cada arquivo ao balancear,
    # usei um ID genérico para a ave toda, MAS mantive a rastreabilidade
    # pelo CSV que acabou de ser preenchido!

    janelas_originais = extrair_janelas_originais(gravacoes)
    janelas_balanceadas = balancear_dataset(janelas_originais)

    if janelas_balanceadas:
        print(f"   -> Salvando {len(janelas_balanceadas)} espectrogramas...")

        # Enviamos um nome genérico como ID (ex: Turdus_rufiventris_dataset)
        # para a função não dar erro ao salvar as imagens
        id_generico = nome_cientifico.replace(" ", "_")
        salvar_espectrogramas(janelas_balanceadas, nome_cientifico, id_generico)

## FLUXO PRINCIPAL

In [ ]:
if __name__ == "__main__":
    # Carrega as tuplas (Nome Cientifico, Nome Popular)
    especies_alvo = carregar_ou_buscar_especies(quantidade=250, minimo_gravacoes=45)
    print(f"\nIniciando o processamento de {len(especies_alvo)} espécies.")

    # Desempacota as duas colunas do CSV durante o loop
    for nome_cientifico, nome_popular in especies_alvo:
        # A função que baixa os áudios só precisa do nome científico
        processar_especie(nome_cientifico)

    print("\nPROCESSO TOTAL CONCLUÍDO!")

Mapeando aves com no mínimo 45 gravações...


Lendo base de dados:  72%|███████▏  | 206/287 [05:34<02:28,  1.83s/it]

In [ ]:
# Salva tudo em um CSV!
df_final = pd.DataFrame(TABELA_METADADOS_GLOBAL)
df_final.to_csv("dataset_sabia_metadados.csv", index=False)
print("Dataset de Imagens e CSV de Metadados finalizados!")